# Initialization and Reproducibility

The computational environment for the experiments is initialized, including the required libraries, working directories, execution device, and random seeds to ensure reproducibility.

In [ ]:
import os
import json
import time
import random
import shutil
from pathlib import Path
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch.utils.data import TensorDataset, DataLoader
from nflows import transforms, distributions, flows

In [ ]:
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [ ]:
# Define main paths
DATA_BASE = Path("/MUON-SPECTRUM-CNF/processed_data") # Adjust paths as needed
WORK_BASE = Path("/MUON-SPECTRUM-CNF/results") # Adjust paths as needed
RANDOM_SEARCH_DIR = WORK_BASE / "Random_search"
RANDOM_SEARCH_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH = WORK_BASE / "Random_search.zip"

# Select available device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Define base seed
BASE_SEED = 42
os.environ["PYTHONHASHSEED"] = str(BASE_SEED)

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

def reset_for_trial(base_seed: int = BASE_SEED):
    seed_everything(base_seed)


seed_everything(BASE_SEED)

# Configure CuDNN behavior
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = False

# Site Definition and Data Loading

The training and validation sites are established, each site is associated with its corresponding input file, and the datasets are loaded into unified DataFrame structures. A site label is added to each row to preserve the origin of every event after concatenation.

In [ ]:
from src.preprocessing import (
    load_sites,
    downsample_per_site_joint,
    make_loader,
)

In [ ]:
# Define cities 
train_files = {
    "lond":  "lond_3600.csv",   # h=24
    "iqa":   "iqa_3600.csv",    # h=34
    "chia":  "chia_3600.csv",   # h=522
    "bga":   "bga_3600.csv",    # h=956
    "bra":   "bra_3600.csv",    # h=1172
    "mge":   "mge_3600.csv",    # h=1450
    "sudb":  "sudb_3600.csv",   # h=2100
    "pamp":  "pamp_3600.csv",   # h=2342
    "casp":  "casp_3600.csv",   # h=2600
    "serb":  "serb_3600.csv",   # h=2750
    "ber":   "ber_3600.csv",    # h=3450
    "lpb":   "lpb_3600.csv",    # h=3630
    "and":   "and_3600.csv",    # h=4200
    "sng":   "sng_3600.csv",    # h=4550
    "cha":   "cha_3600.csv",    # h=5230
}


val_files = {
    "lim":   "lim_3600.csv",    # h=150
    "gua":   "gua_3600.csv",    # h=1490
    "quie":  "quie_3600.csv",   # h=2850
    "ima":   "ima_3600.csv",    # h=4600 
    "ata":   "ata_3600.csv",    # h=5105
}

train_sites = list(train_files.keys())
val_sites = list(val_files.keys())

print("Train sites:", train_sites)
print("Val sites:", val_sites)

In [ ]:
# Load training and validation data
df_train = load_sites(train_sites, train_files, DATA_BASE)
df_val   = load_sites(val_sites, val_files, DATA_BASE)

print("Raw train rows:", len(df_train))
print("Raw val rows:", len(df_val))

# Data preprocessing

The main preprocessing settings are defined, and the data preparation pipeline used before training is implemented. First, a joint per-site downsampling strategy is applied to reduce site imbalance while preserving the joint distribution of energy and zenith angle.

In [19]:
# Data preprocessing settings
TARGET_PER_SITE_TRAIN = 370_000
N_BINS = 100
EPS = 1e-12
EPS_ANG = 1e-6
context_cols = ["h", "bx", "bz"]
RNG = np.random.default_rng(42)

In [ ]:
# Global energy bins from raw training data
logE_train_all = np.log(df_train["energy"].to_numpy(dtype=float))
LogEmin = logE_train_all.min()
LogEmax = logE_train_all.max()
binsE = np.linspace(LogEmin, LogEmax + 1e-8, N_BINS + 1)

# Angle bins in [0, 90]
binsA = np.linspace(0.0, 90.0 + 1e-8, N_BINS + 1)

# Apply joint downsampling to the training set
df_train = downsample_per_site_joint(
    df_train,
    TARGET_PER_SITE_TRAIN,
    binsE,
    binsA,
    RNG
)

print("Train after joint downsample:", len(df_train))
print(df_train["site"].value_counts().head())

The conditional variables are normalized with a MinMax transformation computed from the training set, and the same statistics are later applied to the validation set. Energy is first transformed to logarithmic scale and then normalized. For the angular component, the variable 𝜇=1−cos(𝜃).

In [ ]:
stats = {}

# MinMax normalization for context variables
for col in context_cols:
    x = df_train[col].to_numpy(dtype=float)
    x_min = float(x.min())
    x_max = float(x.max())
    x_range = x_max - x_min

    df_train[col + "_mm"] = (df_train[col] - x_min) / (x_range + EPS)
    df_val[col + "_mm"]   = (df_val[col]   - x_min) / (x_range + EPS)

    stats[col] = {"min": x_min, "max": x_max, "range": x_range}

# Log + MinMax normalization for energy
logE_train = np.log(df_train["energy"].to_numpy(dtype=float))
z_min = float(logE_train.min())
z_max = float(logE_train.max())
z_range = z_max - z_min

df_train["logE_mm"] = (logE_train - z_min) / (z_range + EPS)
stats["logE"] = {"min": z_min, "max": z_max, "range": z_range}

# Apply training statistics to validation data
logE_val = np.log(df_val["energy"].to_numpy(dtype=float))
df_val["logE_mm"] = (logE_val - z_min) / (z_range + EPS)

# Angular variable for the model
mu_train = 1.0 - np.cos(np.deg2rad(df_train["theta"].to_numpy(dtype=float)))
mu_val   = 1.0 - np.cos(np.deg2rad(df_val["theta"].to_numpy(dtype=float)))

df_train["mu"] = np.clip(mu_train, EPS_ANG, 1.0 - EPS_ANG)
df_val["mu"]   = np.clip(mu_val,   EPS_ANG, 1.0 - EPS_ANG)

stats["mu"] = {"min": 0.0, "max": 1.0, "range": 1.0}

stats

In [ ]:
# Training tensors
X_train = torch.from_numpy(df_train[["logE_mm", "mu"]].to_numpy(dtype=np.float32))
C_train = torch.from_numpy(df_train[[c + "_mm" for c in context_cols]].to_numpy(dtype=np.float32))
train_tensor = TensorDataset(X_train, C_train)

# Validation tensors
X_val = torch.from_numpy(df_val[["logE_mm", "mu"]].to_numpy(dtype=np.float32))
C_val = torch.from_numpy(df_val[[c + "_mm" for c in context_cols]].to_numpy(dtype=np.float32))
val_tensor = TensorDataset(X_val, C_val)

# CNFs architecture and training

The fixed training hyperparameters, the conditional flow architecture, and the optimization procedure are established. The model is constructed as a two-dimensional conditional normalizing flow, where the target variables are the normalized logarithmic energy and the transformed angular variable μ. Training is performed by minimizing the negative log-likelihood over the training set, while the validation loss is evaluated after each epoch.

In [ ]:
# Training configuration
EPOCHS = 20
N_TRIALS = 20
LR_FIXED = 3e-4

# Hyperparameter search space
space_hidden = [192,256,320]
space_layers = [5,6,7]
space_bins   = [3,4,5]
space_tail   = [4,6,8]

In [ ]:
from src.model import (
    build_flow,
    val_flow,
    train_flow,
)

# Validation utilities and inference helpers

The necessary tools for validation and inference are integrated, including the forward and inverse transformations, the spectral error metrics used to compare the model predictions with the reference distributions, and the truncated set sampling procedure.

In [ ]:
# Validation settings
Y_MIN = 0.0
Y_MAX = 0.86
MU_MIN = 1e-6
MU_MAX = 1.0 - 1e-6
SAMPLE_BATCH = 1000

# Histogram bins and validation seeds
bins_E = np.logspace(-2, 4, 50)
bins_theta = np.linspace(0.0, 90, 91)
VAL_SEEDS = [5, 503, 1001, 1501, 1999]

In [ ]:
from src.validation_helpers import (
    logE_to_mm,
    mm_to_energy,
    add_mu,
    mu_to_theta,
    con_mm,
    energy_spectrum,
    angle_spectrum,
    abs_relative_error_per_bin,
    weighted_relative_error_per_bin,
    masked_mean,
    range_mean,
    sample_model_joint,
)

Although normalization was calculated from the entire training set, individual sites do not necessarily cover the entire interval [0,1]; therefore, sampling was restricted to y ≤ 0.86, which covers the region where most of the site-level data is concentrated.

# Plotting utilities for spectral comparison

The plotting style adopted for validation was established, along with the functions used to save the comparisons of the energy and angular spectra. Each figure includes the reference spectrum, the mean model prediction, the associated ±1σ band across validation seeds, and the contribution of the weighted relative error shown in a lower panel.

In [ ]:
# Global plotting style
plt.rcParams.update({
    "font.size": 18,
    "axes.titlesize": 18,
    "axes.labelsize": 16,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "legend.fontsize": 13,
    "lines.linewidth": 2.5,
    "grid.linewidth": 0.9,
})

In [ ]:
from src.plotting import (
    save_energy_plot,
    save_angle_plot,
)

# Validation-site evaluation and aggregated metrics

For each city, the data are transformed into the model space, truncated to the valid normalized energy range, and used to define the site-level conditioning context. Multiple sampling seeds are used to generate model realizations, from which spectral summaries, interval-wise errors, and aggregate validation metrics are computed. The resulting figures and metric tables are saved for all validation sites.

In [ ]:
def evaluate_city(flow, city, data_base, stats, device, val_seeds,
                  y_min=Y_MIN, y_max=Y_MAX, mu_min=MU_MIN, mu_max=MU_MAX,
                  sample_batch=SAMPLE_BATCH, bins_E=bins_E, bins_theta=bins_theta):
    """
    Evaluate the model on a single validation site and return summary metrics 
    together with the data required for plotting.
    """
    # Load the site data and map all variables to the representation used in validation.
    df_city = pd.read_csv(data_base / f"{city}_3600.csv")
    df_city = logE_to_mm(df_city, stats)
    df_city = add_mu(df_city)
    df_city = con_mm(df_city, stats, context_cols)

   # Keep only rows inside the normalized energy interval used during truncated sampling.
    mask = df_city["logE_mm"].between(y_min, y_max)
    df_city = df_city.loc[mask].reset_index(drop=True)

    if len(df_city) == 0:
        raise ValueError(f"{city}: no rows after energy truncation.")

    # The conditioning vector is site-level, so one row is enough to define it.
    c_joint = torch.tensor(
        df_city.iloc[0][[c + "_mm" for c in context_cols]].to_numpy(np.float32),
        device=device
    ).view(1, -1)

    E_real = df_city["energy"].to_numpy(dtype=np.float32)
    theta_real = df_city["theta"].to_numpy(dtype=np.float32)

    # Reference spectra from the validation data.
    bin_centers_E, spec_real_E = energy_spectrum(E_real, bins_E)
    bin_centers_A, spec_real_A = angle_spectrum(theta_real, bins_theta)

    # Per-seed outputs are stored first and aggregated afterwards.
    spec_models_E = []
    rank_errs_E = []
    plot_errs_E = []

    spec_models_A = []
    rank_errs_A = []
    plot_errs_A = []

    for s in val_seeds:
        # Repeating the sampling with different seeds provides a stability estimate.
        seed_everything(int(s))

        y_joint = sample_model_joint(
            flow=flow,
            c=c_joint,
            y_min=y_min,
            y_max=y_max,
            mu_min=mu_min,
            mu_max=mu_max,
            batch=sample_batch,
            sample=len(df_city)
        )

        y_logE_mm = y_joint[:, 0]
        y_mu      = y_joint[:, 1]

        # Map sampled values back to the physical variables used for the spectra.
        E_samp = mm_to_energy(y_logE_mm, stats)
        theta_samp = mu_to_theta(y_mu)

        _, spec_model_E = energy_spectrum(E_samp, bins_E)
        _, spec_model_A = angle_spectrum(theta_samp, bins_theta)

        spec_models_E.append(spec_model_E)
        spec_models_A.append(spec_model_A)

        # Absolute relative errors are used for ranking and summary metrics.
        rank_errs_E.append(abs_relative_error_per_bin(spec_real_E, spec_model_E))
        plot_errs_E.append(weighted_relative_error_per_bin(spec_real_E, spec_model_E))

        # Weighted relative errors are used for visualization.
        rank_errs_A.append(abs_relative_error_per_bin(spec_real_A, spec_model_A))
        plot_errs_A.append(weighted_relative_error_per_bin(spec_real_A, spec_model_A))

    spec_models_E = np.vstack(spec_models_E)
    rank_errs_E   = np.vstack(rank_errs_E)
    plot_errs_E   = np.vstack(plot_errs_E)

    spec_models_A = np.vstack(spec_models_A)
    rank_errs_A   = np.vstack(rank_errs_A)
    plot_errs_A   = np.vstack(plot_errs_A)

    # Aggregate the per-seed results into mean spectra and dispersion bands.
    spec_mean_E = np.mean(spec_models_E, axis=0)
    spec_std_E  = np.std(spec_models_E, axis=0)
    rank_mean_E = np.nanmean(rank_errs_E, axis=0)
    plot_mean_E = np.nanmean(plot_errs_E, axis=0)
    plot_std_E  = np.nanstd(plot_errs_E, axis=0)

    spec_mean_A = np.mean(spec_models_A, axis=0)
    spec_std_A  = np.std(spec_models_A, axis=0)
    rank_mean_A = np.nanmean(rank_errs_A, axis=0)
    plot_mean_A = np.nanmean(plot_errs_A, axis=0)
    plot_std_A  = np.nanstd(plot_errs_A, axis=0)

    # Summary metrics are reported globally and by predefined spectral intervals.
    energy_metrics = {
        "city": city,
        "rel_gl_mean": masked_mean(rank_mean_E),
        "rel_1e-1_1e2": range_mean(rank_mean_E, bin_centers_E, 1e-1, 1e2),
        "rel_1e2_1e3": range_mean(rank_mean_E, bin_centers_E, 1e2, 1e3),
        "rel_1e3_1e4": range_mean(rank_mean_E, bin_centers_E, 1e3, 1e4),
    }

    angle_metrics = {
        "city": city,
        "rel_gl_mean": masked_mean(rank_mean_A),
        "rel_0_30": range_mean(rank_mean_A, bin_centers_A, 0.0, 30.0),
        "rel_30_60": range_mean(rank_mean_A, bin_centers_A, 30.0, 60.0),
        "rel_60_90": range_mean(rank_mean_A, bin_centers_A, 60.0, 90.0 + 1e-9),
    }

    # Plot payload keeps only the arrays needed to build the validation figures.
    plot_payload = {
        "bin_centers_E": bin_centers_E,
        "spec_real_E": spec_real_E,
        "spec_mean_E": spec_mean_E,
        "spec_std_E": spec_std_E,
        "plot_mean_E": plot_mean_E,
        "plot_std_E": plot_std_E,
        "bin_centers_A": bin_centers_A,
        "spec_real_A": spec_real_A,
        "spec_mean_A": spec_mean_A,
        "spec_std_A": spec_std_A,
        "plot_mean_A": plot_mean_A,
        "plot_std_A": plot_std_A,
    }

    return energy_metrics, angle_metrics, plot_payload


def evaluate_validation_set(flow, val_sites, data_base, stats, device, val_dir):
    """
    Evaluate the model on all validation sites, save plots and metric tables,
    and return the aggregated validation summary.
    """
    val_dir.mkdir(parents=True, exist_ok=True)

    rows_energy = []
    rows_angle = []

    for city in val_sites:
        energy_metrics, angle_metrics, plot_payload = evaluate_city(
            flow=flow,
            city=city,
            data_base=data_base,
            stats=stats,
            device=device,
            val_seeds=VAL_SEEDS
        )

        rows_energy.append(energy_metrics)
        rows_angle.append(angle_metrics)

        save_energy_plot(
            city=city,
            bin_centers_E=plot_payload["bin_centers_E"],
            spec_real_E=plot_payload["spec_real_E"],
            spec_mean_E=plot_payload["spec_mean_E"],
            spec_std_E=plot_payload["spec_std_E"],
            contrib_mean_E=plot_payload["plot_mean_E"],
            contrib_std_E=plot_payload["plot_std_E"],
            save_path=val_dir / f"{city}_energy.png"
        )

        save_angle_plot(
            city=city,
            bin_centers_A=plot_payload["bin_centers_A"],
            spec_real_A=plot_payload["spec_real_A"],
            spec_mean_A=plot_payload["spec_mean_A"],
            spec_std_A=plot_payload["spec_std_A"],
            contrib_mean_A=plot_payload["plot_mean_A"],
            contrib_std_A=plot_payload["plot_std_A"],
            save_path=val_dir / f"{city}_angle.png"
        )

    metrics_energy_df = pd.DataFrame(rows_energy)
    metrics_angle_df = pd.DataFrame(rows_angle)

    # Save per-city validation metrics for later analysis and reporting.
    metrics_energy_df.to_csv(val_dir / "metrics_energy.csv", index=False)
    metrics_angle_df.to_csv(val_dir / "metrics_angle.csv", index=False)

    # Global validation scores are computed as averages across validation sites.
    rel_gl_mean_energy = float(metrics_energy_df["rel_gl_mean"].mean())
    rel_gl_mean_angle = float(metrics_angle_df["rel_gl_mean"].mean())
    rel_gl_mean_total = float((rel_gl_mean_energy + rel_gl_mean_angle) / 2.0)

    return {
        "metrics_energy_df": metrics_energy_df,
        "metrics_angle_df": metrics_angle_df,
        "rel_gl_mean_energy": rel_gl_mean_energy,
        "rel_gl_mean_angle": rel_gl_mean_angle,
        "rel_gl_mean_total": rel_gl_mean_total,
    }

This section defines the helper functions used to generate the hyperparameter configurations evaluated during the random search and to store the resulting outputs in JSON format.

In [ ]:
def build_trial_table(n_trials, base_seed=BASE_SEED):
    """
    Create a reproducible list of randomly selected hyperparameter
    configurations for random search.
    """
    rng = np.random.default_rng(base_seed)

    all_configs = [
        {
            "hidden_features": int(h),
            "num_layers": int(l),
            "num_bins": int(b),
            "tail_bound": float(t),
        }
        for h, l, b, t in itertools.product(space_hidden, space_layers, space_bins, space_tail)
    ]

    if n_trials > len(all_configs):
        raise ValueError(f"N_TRIALS={n_trials} exceeds total unique combinations={len(all_configs)}")

    # Sample unique configurations without replacement
    chosen_idx = rng.choice(len(all_configs), size=n_trials, replace=False)

    rows = []
    for trial, idx in enumerate(chosen_idx):
        rows.append({
            "trial": trial,
            **all_configs[int(idx)]
        })

    return rows

In [ ]:
def save_json(path, obj):
    """
    Save a Python object as a JSON file, creating the parent directory if needed.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

# Random Search Execution and Output Export

This section stores the global metadata, runs the random search over the selected hyperparameter configurations, evaluates each trained model on the validation sites, and exports the final results.

In [ ]:
# Save global metadata
split_sites = {
    "train_sites": train_sites,
    "val_sites": val_sites,
}

save_json(RANDOM_SEARCH_DIR / "split_sites.json", split_sites)
save_json(RANDOM_SEARCH_DIR / "stats.json", stats)

trial_table = build_trial_table(N_TRIALS, base_seed=BASE_SEED)

metrics_rows = []

for trial_cfg in trial_table:
    trial_idx = int(trial_cfg["trial"])
    trial_name = f"trial_{trial_idx:02d}"
    trial_dir = RANDOM_SEARCH_DIR / trial_name
    val_dir = trial_dir / "val"

    trial_dir.mkdir(parents=True, exist_ok=True)
    val_dir.mkdir(parents=True, exist_ok=True)

    # Reset seeds before each trial
    reset_for_trial(BASE_SEED)

    # Create data loaders
    train_loader = make_loader(
        train_tensor,
        batch_size=4096,
        shuffle=True,
        num_workers=2
    )

    val_loader = make_loader(
        val_tensor,
        batch_size=8192,
        shuffle=False,
        num_workers=2
    )

    print("=" * 90)
    print(f"TRIAL {trial_idx:02d}")
    print({
        **trial_cfg,
        "base_seed": BASE_SEED,
        "lr": LR_FIXED,
        "epochs": EPOCHS
    })

    # Build and train the model
    flow = build_flow(
        hidden_features=trial_cfg["hidden_features"],
        num_layers=trial_cfg["num_layers"],
        num_bins=trial_cfg["num_bins"],
        tail_bound=trial_cfg["tail_bound"]
    )

    flow, history = train_flow(
        flow=flow,
        train_loader=train_loader,
        val_loader=val_loader,
        device=device,
        epochs=EPOCHS,
        lr=LR_FIXED
    )

    # Save trained weights
    torch.save(flow.state_dict(), trial_dir / "model.pt")

    # Save training history
    save_json(trial_dir / "history.json", history)

    # Save trial configuration
    config = {
        "trial": trial_idx,
        "trial_name": trial_name,
        "base_seed": BASE_SEED,
        "epochs": EPOCHS,
        "lr": LR_FIXED,
        "hidden_features": trial_cfg["hidden_features"],
        "num_layers": trial_cfg["num_layers"],
        "num_bins": trial_cfg["num_bins"],
        "tail_bound": trial_cfg["tail_bound"],
        "val_seeds": VAL_SEEDS,
        "y_min": Y_MIN,
        "y_max": Y_MAX,
        "mu_min": MU_MIN,
        "mu_max": MU_MAX,
        "sample_batch": SAMPLE_BATCH,
    }
    save_json(trial_dir / "config.json", config)

    # Run validation across all validation sites
    val_out = evaluate_validation_set(
        flow=flow.to(device).eval(),
        val_sites=val_sites,
        data_base=DATA_BASE,
        stats=stats,
        device=device,
        val_dir=val_dir
    )

    nll_final = float(history[-1]["val_nll"])

    # Store summary metrics for this trial
    metrics_rows.append({
        "trial": trial_idx,
        "hidden_features": trial_cfg["hidden_features"],
        "num_layers": trial_cfg["num_layers"],
        "num_bins": trial_cfg["num_bins"],
        "tail_bound": trial_cfg["tail_bound"],
        "nll_final": nll_final,
        "rel_gl_mean_energy": val_out["rel_gl_mean_energy"],
        "rel_gl_mean_angle": val_out["rel_gl_mean_angle"],
        "rel_gl_mean_total": val_out["rel_gl_mean_total"],
    })

    # Release GPU memory before the next trial
    del flow
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# Export metrics table
metrics_df = pd.DataFrame(metrics_rows).sort_values("trial").reset_index(drop=True)
metrics_df.to_excel(RANDOM_SEARCH_DIR / "metrics.xlsx", index=False)

print(metrics_df)


A compressed .zip archive is created containing the full output directory generated during the random-search runs. Packaging the results into a single file simplifies download, storage, and later reuse of the trained models, validation figures, and metric tables.

In [ ]:
# Compress final outputs
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

shutil.make_archive(
    base_name=str(ZIP_PATH.with_suffix("")),
    format="zip",
    root_dir=RANDOM_SEARCH_DIR.parent,
    base_dir=RANDOM_SEARCH_DIR.name
)

print("Random search finished.")
print("Folder:", RANDOM_SEARCH_DIR)
print("ZIP:", ZIP_PATH)